# IntelliAdmit — SOP Fine-tuning (phi-2, no quantization)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `sops.jsonl` using the Files panel (navigate to `/content/` first)
3. Add `HF_TOKEN` in Secrets panel (key icon) with **Notebook access ON**

Uses `microsoft/phi-2` (2.7B) in fp16 — fits in 16GB T4 with 8GB to spare.
No bitsandbytes, no unsloth, no quantization = zero dependency conflicts.

In [ ]:
# Cell 1 — GPU check
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2 — Install / upgrade required packages
%pip install -q -U peft trl accelerate datasets huggingface_hub torchao
print('Done — if you see a restart prompt, click Restart and re-run from Cell 1.')

In [ ]:
# Cell 3 — HuggingFace token
from google.colab import userdata
import os

HF_TOKEN = userdata.get('Germany')
HF_REPO  = 'allisamhitha/intelliadmit-sop-lora'

if not HF_TOKEN:
    raise ValueError('Add your HF token to Colab Secrets with key name "Germany" and Notebook access ON')

os.environ['HF_TOKEN'] = HF_TOKEN
print(f'Token loaded. Adapter will be pushed to: {HF_REPO}')

In [ ]:
# Cell 4 — Load SOP dataset
import json
from pathlib import Path
from datasets import Dataset

DATASET_PATH = '/content/sops.jsonl'
if not Path(DATASET_PATH).exists():
    # Check if uploaded to wrong location
    import glob
    found = glob.glob('/**/sops.jsonl', recursive=True)
    if found:
        import shutil
        shutil.copy(found[0], DATASET_PATH)
        print(f'Moved from {found[0]}')
    else:
        raise FileNotFoundError('Upload sops.jsonl via the Files panel (navigate to /content/ first)')

# phi-2 prompt format: Instruct / Output
TEMPLATE = 'Instruct: {instruction}\nOutput: {response}'

records = []
for line in Path(DATASET_PATH).read_text(encoding='utf-8').splitlines():
    if not line.strip():
        continue
    row = json.loads(line)
    instruction = (
        f"Student profile: {json.dumps(row['profile'])}\n"
        f"Target program: {row['program']}\n"
        'Write a Statement of Purpose in German academic English.'
    )
    records.append({'text': TEMPLATE.format(instruction=instruction, response=row['response'])})

dataset = Dataset.from_list(records)
print(f'Loaded {len(dataset)} training examples')
print('Sample (first 200 chars):')
print(dataset[0]['text'][:200])

In [ ]:
# Cell 5 — Load phi-2 in fp16 (no quantization, no bitsandbytes)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = 'microsoft/phi-2'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in fp16 (takes ~1 min)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded!  VRAM: {used:.1f} GB / {total:.1f} GB  ({total-used:.1f} GB free)')

In [ ]:
# Cell 6 — Attach LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'dense'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5% of 2.7B = ~13M trainable params

In [ ]:
# Cell 7 — Train
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

OUTPUT_DIR = '/content/intelliadmit-sop-lora'

# Tokenize only — do NOT set labels manually.
# DataCollatorForLanguageModeling(mlm=False) pads input_ids per batch,
# then sets labels = input_ids with -100 on padding tokens automatically.
def tokenize(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=1024,
        padding=False,
    )

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    optim='adamw_torch',
    report_to='none',
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=args,
    data_collator=collator,
)

print(f'Training: {len(tokenized_dataset)} examples, 3 epochs, fp16')
print('ETA: ~10-15 minutes on T4\n')
trainer.train()
print('Training complete!')

In [ ]:
# Cell 8 — Save adapter locally
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
total_mb = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f))
               for f in os.listdir(OUTPUT_DIR)) / 1e6
print(f'Adapter saved to {OUTPUT_DIR} ({total_mb:.1f} MB)')

In [ ]:
# Cell 9 — Push to HuggingFace Hub
from huggingface_hub import HfApi

HfApi().create_repo(repo_id=HF_REPO, token=HF_TOKEN, exist_ok=True)
trainer.model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

print(f'Pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# Cell 10 — Quick sanity check
model.eval()
prompt = (
    'Instruct: Student profile: {"degree": "B.Tech Computer Science", "cgpa": 8.1, '
    '"target_field": "Computer Science", "application_level": "masters"}\n'
    'Target program: TU Munich MSc Informatics\n'
    'Write a Statement of Purpose in German academic English.\nOutput:'
)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, temperature=0.7,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(text[:500])